<a href="https://colab.research.google.com/github/AngeloSorte/Global_Meteorite_and_NEO_Risk_Study/blob/angelosorte.github.io/Global_Meteorite_Fall_Analysis_and_Mapping_(1500_2024).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [11]:
import pandas as pd
import numpy as np

# Load the dataset (assuming it is in the Meteorite_Landings.csv file)
df = pd.read_csv('/content/Meteorite_Landings.csv')

# Show the first rows to understand how it is made
print(df.head())

# Take only the columns that interest us
df = df[['year', 'reclat', 'reclong']]

# Remove rows without coordinates or with null values
df = df.dropna(subset=['reclat', 'reclong', 'year'])

# Convert year to integer (some years can be in string or float format)
df['year'] = df['year'].astype(int)

# Consider only meteorites that fell in the last 500 years (from 1500 onwards)
df = df[df['year'] >= 1500]

# Create a geographic grid by dividing lat and long into 10 degree cells
df['lat_bin'] = (df['reclat'] // 10) * 10
df['long_bin'] = (df['reclong'] // 10) * 10

# Count how many falls there are in each cell
counts = df.groupby(['lat_bin', 'long_bin']).size().reset_index(name='count')

# Sort by number of falls descending
counts = counts.sort_values(by='count', ascending=False)

print(counts.head(10))


       name     id nametype     recclass  mass (g)  fall    year    reclat  \
0    Aachen    1.0    Valid           L5      21.0  Fell  1880.0  50.77500   
1    Aarhus    2.0    Valid           H6     720.0  Fell  1951.0  56.18333   
2      Abee    6.0    Valid          EH4  107000.0  Fell  1952.0  54.21667   
3  Acapulco   10.0    Valid  Acapulcoite    1914.0  Fell  1976.0  16.88333   
4   Achiras  370.0    Valid           L6     780.0  Fell  1902.0 -33.16667   

     reclong           GeoLocation  
0    6.08333     (50.775, 6.08333)  
1   10.23333  (56.18333, 10.23333)  
2 -113.00000    (54.21667, -113.0)  
3  -99.90000     (16.88333, -99.9)  
4  -64.95000   (-33.16667, -64.95)  
     lat_bin  long_bin  count
8      -90.0     160.0   5588
14     -80.0     150.0   5193
72       0.0       0.0   4517
13     -80.0      70.0   2494
94      10.0      50.0   2049
11     -80.0      20.0   1509
109     20.0      10.0   1441
4      -90.0     -70.0    908
15     -80.0     160.0    828
108     2

In [12]:
# Step 0: Install folium (run this cell only once)
!pip install folium

# Step 1: Import libraries
import pandas as pd
import folium
from folium.plugins import HeatMap

# Step 2: Load the dataset
# Make sure 'Meteorite_Landings.csv' is uploaded to Colab or available in the current folder
df = pd.read_csv('Meteorite_Landings.csv')

# Step 3: Data cleaning
# Keep only columns of interest: year, reclat (latitude), reclong (longitude)
df = df[['year', 'reclat', 'reclong']]

# Drop rows with missing values in these columns
df = df.dropna(subset=['year', 'reclat', 'reclong'])

# Convert year to integer (some may be floats or strings)
df['year'] = df['year'].astype(int)

# Filter data for meteorites fallen since 1500 (to focus on recent history)
df = df[df['year'] >= 1500]

# Step 4: Group data into geographic bins (10° x 10°)
# Bin latitude and longitude into 10 degree intervals
df['lat_bin'] = (df['reclat'] // 10) * 10
df['long_bin'] = (df['reclong'] // 10) * 10

# Count the number of meteorites per grid cell
counts = df.groupby(['lat_bin', 'long_bin']).size().reset_index(name='count')

# Step 5: Prepare data points for the heatmap
# HeatMap expects a list of [lat, lon, weight]
heat_data = [
    [row['lat_bin'] + 5, row['long_bin'] + 5, row['count']]  # center points of each bin (+5 for center)
    for index, row in counts.iterrows()
]

# Step 6: Create a Folium map centered roughly at the center of the world
m = folium.Map(location=[0, 0], zoom_start=2)

# Step 7: Add HeatMap layer with weights (count of meteorites)
HeatMap(heat_data, radius=25, max_zoom=6).add_to(m)

# Step 8: Show the map
m
